# Reports: Word, Excel, PDF

Runs **offline** with the bundled python-docx, XlsxWriter, pypdf, Jinja2 and tabulate (in Jupyter: `pip install python-docx xlsxwriter pypdf jinja2 tabulate`). The files appear in the Files panel – open them with ↗ in the document editor, the spreadsheet or the PDF viewer.

In [ ]:
import pandas as pd
from tabulate import tabulate

sales = pd.DataFrame({'month': ['Jan', 'Feb', 'Mar'], 'revenue': [1200, 1350, 1100]})
print(tabulate(sales, headers='keys', tablefmt='github', showindex=False))

In [ ]:
import docx
from jinja2 import Template

text = Template('Revenue in {{ months }} months: {{ total }} EUR.').render(months=len(sales), total=sales.iloc[:, 1].sum())
doc = docx.Document()
doc.add_heading('Quarterly report', 0)
doc.add_paragraph(text)
table = doc.add_table(rows=1, cols=2)
table.rows[0].cells[0].text, table.rows[0].cells[1].text = sales.columns
for m, r in sales.itertuples(index=False):
    row = table.add_row().cells
    row[0].text, row[1].text = m, str(r)
doc.save('report.docx')
text

In [ ]:
with pd.ExcelWriter('report.xlsx', engine='xlsxwriter') as writer:
    sales.to_excel(writer, sheet_name='Sales', index=False)
    sheet = writer.sheets['Sales']
    chart = writer.book.add_chart({'type': 'column'})
    chart.add_series({'categories': ['Sales', 1, 0, 3, 0], 'values': ['Sales', 1, 1, 3, 1]})
    sheet.insert_chart('D2', chart)
print('written')

In [ ]:
import io
import matplotlib.pyplot as plt
from pypdf import PdfReader, PdfWriter

# two one-page PDFs (from matplotlib), merged into one
writer = PdfWriter()
for kind in ('bar', 'line'):
    buf = io.BytesIO()
    sales.plot(x=sales.columns[0], kind=kind, legend=False).figure.savefig(buf, format='pdf')
    plt.close('all')
    writer.append(PdfReader(buf))
writer.write('charts.pdf')
len(PdfReader('charts.pdf').pages)